Project: /data-manager/api/_project.yaml
Book: /data-manager/api/_book.yaml

<style>
  devsite-code .tfo-notebook-code-cell-output {
    max-height: 300px;
    overflow: auto;
    background: rgba(255, 247, 237, 1);  /* light orange bg */
  }
  
  devsite-code .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
    background: rgba(255, 247, 237, .7);
  }
  
  devsite-code[dark-code] .tfo-notebook-code-cell-output {
    background: rgba(64, 78, 103, 1);  /* dark mode slate */
  }
  
  devsite-code[dark-code] .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
    background: rgba(64, 78, 103, .7);
  }
  
  .devsite-table-wrapper .tfo-notebook-buttons {
    display: inline-block;
    margin-left: 3px;
    width: auto;
    border: 0;
  }
  
  .tfo-notebook-buttons tr {
    background: 0;
    border: 0;
  }
  
  .tfo-notebook-buttons td {
    padding-left: 0;
    padding-right: 20px;
    border: 0;
  }
  
  .tfo-notebook-buttons {
    --tfo-notebook-buttons-box-shadow: 0 1px 2px 0 rgba(60, 64, 67, .3), 0 1px 3px 1px rgba(60, 64, 67, .15);
  }
  
  .tfo-notebook-buttons a,
  .tfo-notebook-buttons :link,
  .tfo-notebook-buttons :visited {
    border-radius: 8px;
    box-shadow: var(--tfo-notebook-buttons-box-shadow);
    color: #202124;
    padding: 12px 24px;
    transition: box-shadow 0.2s;
    text-decoration: none;
    display: flex;
    align-items: center;
  }
  
  .tfo-notebook-buttons a:hover,
  .tfo-notebook-buttons a:focus {
    box-shadow: 0 2px 6px 2px rgba(60, 64, 67, 0.15);
    text-decoration: none;
  }
  
  .tfo-notebook-buttons td > a > img {
    margin-right: 8px;
    width: 32px;
    height: 32px;
  }
  </style>

In [ ]:
# @markdown #### Copyright 2026 Google LLC
# @markdown ##### Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Data Partner: Simplified Testing Workflow

  <table class="tfo-notebook-buttons nocontent" align="left">
    <td>
      <a target="_blank" href="https://colab.research.google.com/github/googleads/data-manager-python/blob/main/notebooks/audience_e2e_data_partner_simplified.ipynb">
      <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
      Run in Google Colab</a>
    </td>
    <td>
      <a target="_blank" href="https://github.com/googleads/data-manager-python/blob/main/notebooks/audience_e2e_data_partner_simplified.ipynb">
      <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
      View source on GitHub</a>
    </td>
  </table>

## Objective
This notebook provides a simplified testing workflow for the Data Manager API as a Data Partner.
It uses a Service Account (or OAuth) to authenticate, assuming the authenticated user has ADMIN access to the Google Ads account.

## Prerequisites

The email address of a
[service account](//cloud.google.com/docs/authentication#service-accounts) that has the required
[access level](https://support.google.com/google-ads/answer/9978556) for each account:

*  The **Admin** access level in the Advertiser account.
*  The **Standard** access level in the Data Partner account.

[Follow these instructions](https://developers.google.com/data-manager/api/devguides/quickstart/set-up-access#service-account) if you don't have a service account.

## Instructions
1. Make a copy of this Colab.
2. Fill in the required account IDs and Service Account details in the Setup cell.
3. Run your copy of the Colab.

### Step 0. Install and Import Packages

In [ ]:
!pip install --upgrade google-ads-datamanager google-auth-oauthlib

In [ ]:
import datetime
from google.colab import userdata
from google.ads import datamanager_v1
from google.api_core import exceptions
import google.auth
from google.protobuf.json_format import MessageToJson

### Step 1. Setup and Configuration

In [ ]:
# @markdown ### Account IDs
# @markdown Provide the advertiser and data partner account IDs below.
# @markdown - **[required]** `login_account_id`: Your Data Partner account.
# @markdown - **[required]** `operating_account_id`: The Google Ads advertiser account that you want to link to your data partner account. This account will own the audience you create as part of this workflow.
login_account_id = ""  # @param {type:"string"}
operating_account_id = ""  # @param {type:"string"}

# @markdown ### Service Account Details
service_account_email = ""  # @param {type:"string"}

# Clean up account IDs by removing hyphens.
login_account_id = login_account_id.replace("-", "")
operating_account_id = operating_account_id.replace("-", "")

GOOGLE_ADS = "GOOGLE_ADS"
DATA_PARTNER = "DATA_PARTNER"
CONSENT_GRANTED = "CONSENT_GRANTED"

### Step 2. Initialize Client

In [ ]:
class DataManagerSDK:
    def __init__(self, creds):
        self.link_service = datamanager_v1.PartnerLinkServiceClient(credentials=creds)
        self.user_list_service = datamanager_v1.UserListServiceClient(credentials=creds)
        self.ingestion_service = datamanager_v1.IngestionServiceClient(credentials=creds)

def initialize_client():
    data_manager_scope = "https://www.googleapis.com/auth/datamanager";
    print("Authenticating with Service Account...")
    auth_command = (
        f"gcloud auth application-default login "
        f"--impersonate-service-account={service_account_email} "
        f"--scopes={data_manager_scope},https://www.googleapis.com/auth/cloud-platform "
        f"--no-browser"
    )
    get_ipython().system(auth_command)
    creds, project = google.auth.default(scopes=[data_manager_scope])
    return DataManagerSDK(creds)

sdk = initialize_client()
print("SDK Services Initialized Successfully")

### Step 3. Create Partner Link

In [ ]:
print(f"3. Creating link for Ads {operating_account_id}..\n")

parent_ads = f"accountTypes/GOOGLE_ADS/accounts/{operating_account_id}"

partner_link_data = datamanager_v1.PartnerLink(
    owning_account=datamanager_v1.ProductAccount(
        account_id=operating_account_id, account_type=GOOGLE_ADS
    ),
    partner_account=datamanager_v1.ProductAccount(
        account_id=login_account_id, account_type=DATA_PARTNER
    ),
)

try:
    link_res = sdk.link_service.create_partner_link(
        parent=parent_ads, partner_link=partner_link_data
    )
    print(f"Link Created: {link_res.partner_link_id}")
    partner_link_id = link_res.partner_link_id
except exceptions.PermissionDenied as e:
    print(f"Link creation failed due to permission denied error: {e}")
    print(
        "Review the 'Prerequisites' section and verify that the service account has the required "
        "access level in the advertiser account."
    )
except Exception as e:
    print(f"Link creation failed or exists: {e}")
    print("\nSearching for existing link...")
    search_res = sdk.link_service.search_partner_links(
        parent=f"accountTypes/DATA_PARTNER/accounts/{login_account_id}"
    )
    # The search result is an iterator
    for link in search_res:
        if link.owning_account.account_id == operating_account_id:
            partner_link_id = link.partner_link_id
            print(f"Found existing Link ID: {partner_link_id}")
            break
    if not partner_link_id:
        print(
            "No existing link found. Review the error details above for more information."
        )

print(
    "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.partnerLinks/create?apix=true"
)
print(f"\nparent: {parent_ads}")
print("\n--- Request JSON Payload ---")
print(MessageToJson(partner_link_data._pb))
print("----------------------------\n")

### Step 4. Create User List

In [ ]:
print(f"4. Creating User List in {operating_account_id}...\n")

parent_userlist = f"accountTypes/GOOGLE_ADS/accounts/{operating_account_id}"

user_list_data = datamanager_v1.UserList(
    display_name=f"Python SDK Audience - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    ingested_user_list_info=datamanager_v1.IngestedUserListInfo(
        upload_key_types=["CONTACT_ID"]
    ),
)

# Creates a dictionary to set headers.
login_account = f"accountTypes/DATA_PARTNER/accounts/{login_account_id}"
linked_account = f"accountTypes/GOOGLE_ADS/accounts/{operating_account_id}"

headers = [
    ("login-account", login_account),
    ("linked-account", linked_account),
]

destination_id = None
try:
    ulist_res = sdk.user_list_service.create_user_list(
        parent=parent_userlist, user_list=user_list_data, metadata=headers
    )

    destination_id = ulist_res.id

    print(f"User List Created!")
    print(f"Destination ID: {destination_id}")
    print(f"Resource Name: {ulist_res.name}")
    print(f"Display Name: {ulist_res.display_name}")

except Exception as e:
    print(f"Failed to create User List: {e}")

print(
    "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.userLists/create?apix=true"
)
print(f"\nparent: {parent_userlist}")

print(f"login-account: {login_account}")
print(f"linked-account: {linked_account}")

print("\n--- Request JSON Payload ---")
print(MessageToJson(user_list_data._pb))
print("----------------------------\n")

### Step 5. Ingest Data

In [ ]:
ingestion_request_id = None

if not destination_id:
    print(
        "Error: destination_id is not set. Cannot ingest data. Please ensure the User List was created successfully in the previous step."
    )
else:
    print(f"5. Ingesting sample data to User List {destination_id}...")

    # Build the Destination object explicitly based on the persona
    destination_obj = datamanager_v1.Destination(
        login_account=datamanager_v1.ProductAccount(
            account_type=DATA_PARTNER, account_id=login_account_id
        ),
        operating_account=datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS, account_id=operating_account_id
        ),
        product_destination_id=str(destination_id),
    )

    # Add the destination_obj directly to the payload
    ingest_payload = datamanager_v1.IngestAudienceMembersRequest(
        consent=datamanager_v1.Consent(
            ad_user_data=CONSENT_GRANTED, ad_personalization=CONSENT_GRANTED
        ),
        encoding=datamanager_v1.Encoding.HEX,
        terms_of_service=datamanager_v1.TermsOfService(
            customer_match_terms_of_service_status="ACCEPTED"
        ),
        validate_only=False,
        audience_members=[
            datamanager_v1.AudienceMember(
                user_data=datamanager_v1.UserData(
                    user_identifiers=[
                        # NOTE: email_address values must be hex-encoded SHA-256
                        # hashes of the normalized email addresses (lowercase, dots
                        # removed from gmail, etc.).
                        # Do NOT use plain text email addresses here.
                        # See Data Manager API documentation for formatting rules:
                        # https://developers.google.com/data-manager/api/devguides/concepts/formatting
                        datamanager_v1.UserIdentifier(
                            email_address="223EBDA6F6889B1494551BA902D9D381DAF2F642BAE055888E96343D53E9F9C4"
                        ),
                        datamanager_v1.UserIdentifier(
                            email_address="F1FCDE379F31F4D446B76EE8F34860ECA2288ADC6B6D6C0FDC56D9EEE75A2FA5"
                        ),
                    ]
                )
            )
        ],
        destinations=[destination_obj],
    )

    try:
        ingest_res = sdk.ingestion_service.ingest_audience_members(
            request=ingest_payload
        )

        ingestion_request_id = ingest_res.request_id
        print(f"Ingestion Submitted!")
        print(f"Request ID: {ingestion_request_id}")

    except Exception as e:
        print(f"Failed to ingest data: {e}")

    print(
        "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/audienceMembers/ingest?apix=true"
    )
    print("\n--- Request JSON Payload ---")
    print(MessageToJson(ingest_payload._pb))
    print("----------------------------\n")

### Step 6. Check Status

In [ ]:
if not ingestion_request_id:
    print("Status check skipped because ingestion request ID is not set.")
else:
    print(f"6. Checking status for Request ID: {ingestion_request_id}...")

    # Note: Headers aren't needed for this request.
    status_req = datamanager_v1.RetrieveRequestStatusRequest(
        request_id=ingestion_request_id
    )

    try:
        status_res = sdk.ingestion_service.retrieve_request_status(
            request=status_req
        )

        print(f"Successfully retrieved status!")

        for dest_status in status_res.request_status_per_destination:
            dest_id = dest_status.destination.product_destination_id
            current_status = dest_status.request_status.name

            print(f"Status for destination {dest_id}: {current_status}")

    except Exception as e:
        print(f"Failed to retrieve status: {e}")

    print(
        "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/requestStatus/retrieve?apix=true"
    )
    print("\n--- Request JSON Payload ---")
    print(MessageToJson(status_req._pb))
    print("----------------------------")

### Step 7. [Optional] Remove Partner Link

In [ ]:
if partner_link_id:
    print(f"\nAttempting to remove partner link {partner_link_id}...")

    link_resource_name = f"accountTypes/GOOGLE_ADS/accounts/{operating_account_id}/partnerLinks/{partner_link_id}"

    delete_req = datamanager_v1.DeletePartnerLinkRequest(
        name=link_resource_name
    )

    try:
        sdk.link_service.delete_partner_link(request=delete_req)
        print("Successfully removed partner link.")

    except Exception as e:
        print(f"Failed to remove partner link: {e}")
else:
    print("\nNo partner_link_id found to delete. Did you run Step 3?")

print(
    "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.partnerLinks/delete?apix=true"
)
print("\n--- Request JSON Payload ---")
print(MessageToJson(delete_req._pb))
print("----------------------------")